# AA-UTE 2026

## P4 - Forecasting con arboles y redes neuronales

Este practico se centra en el uso de modelos de forecasting basados en árboles y redes neuronales. Se trabajará sobre una serie de datos reales correspondiente a la demanda de energía eléctrica en la ciudad de Mercedes, Soriano.

Al finalizar este practico se espera que el estudiante haya podido experimentar con:

- construir features clásicas y targets para el entrenamiento de modelos de forecasating;
- implementar modelos de forecasting con RandomForest y XGBoost de la libreria `sklearn` y `xgboost`;
- implementar un modelo convolucional temporal (TCN) y compararlo con un modelo recurrente (RNN) para forecasting de la libreria `neuralforecast`;

### Trabajo a realizar

Este notebook esta pensado para que al principio se puedan correr las diferentes celdas tal y como están y analizar los resultados. Luego, se proponen algunas modificaciones para que el estudiante pueda experimentar con los modelos y analizar los resultados.

In [1]:
import warnings
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

import xgboost as xgb

from neuralforecast import NeuralForecast
from neuralforecast.models import RNN, TCN
from neuralforecast.losses.pytorch import MSE

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (14, 4)

c:\Users\ut605453\AppData\Local\miniconda3\envs\ts_ute\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-15 12:42:16,894	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-07-15 12:42:17,410	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## Parte 1 - Serie base y extraccion de caracteristicas

Primero cargamos los datos.

In [3]:
# ---------------------------------------------------------------
# 1. Carga de datos
#   - Se cargan los datos que contiene la serie de tiempo a una frecuencia de 15 minutos.
#   - Se convierten los timestamps a formato datetime y se setea como índice del dataframe.
#   - Se remueven columnas innecesarias y se re-muestrea la serie a frecuencia horaria.  
# ---------------------------------------------------------------
path_file = "DATOS_15MIN_MER_trainset-labeled.csv" # puede que sea necesario cambiar la ruta de este archivo.
dta = pd.read_csv(path_file, low_memory=False)
dta["timestamp"] = pd.to_datetime(dta["timestamp"])
dta.set_index("timestamp", inplace=True)
dta = dta.drop(columns=["series", "label"])
dta = dta.resample("h").mean()

dta.head()

,value
timestamp,
2015-04-14 00:00:00+00:00,3.384617
2015-04-14 01:00:00+00:00,2.993700
2015-04-14 02:00:00+00:00,2.807283
2015-04-14 03:00:00+00:00,2.710316
2015-04-14 04:00:00+00:00,2.684633


In [ ]:
print(dta.shape)
dta.tail()

In [ ]:
dta.plot(figsize=(16, 3), title="Serie horaria base (MER)")
plt.xlabel("Tiempo")
plt.ylabel("value")
plt.show()

### EXPERIMENTAR Y DISCUTIR

Visualizar los datos a otras escalas (anual, mensual, semanal), y discutir:

1. ¿Qué patrones se observan a simple vista?
2. ¿Aparecen valores atípicos?

**Respuestas**



In [ ]:
#TODO Plots.

## Parte 2 - Targets y features

En esta sección se trabajará sobre dos problemas de forecasting: predecir la siguiente hora (1h) y predecir la siguiente semana (1w). Para entrenar modelos con árboles es necesario generar un conjunto tabular a partir de diferentes features.

In [ ]:
# El objetivo es predecir el siguiente valor (siguiente hora)
target_1h = dta.shift(-1).copy().dropna()

# El objetivo es predecir todos los valores hasta la misma hora la siguiente semana (168 horas)
target_1w = pd.concat(
    [dta["value"].shift(-k).rename(f"t+{k}") for k in range(1, 169)],
    axis=1
)
target_1w = target_1w.dropna()

target_1h.head(), target_1w.head()

### RESPONDER

En las celda anterior se construyen dos dataframes que serán los targets de los problemas de forecasting. Discutir:

1. ¿Qué representa cada columna de target_1w?
2. ¿Cuál es la diferencia en cantidad de filas entre los datos originales (dta) y los targets? ¿Qué explica esta diferencia?

**Respuestas**


In [ ]:
#TODO: Comparar diferencias.

In [ ]:
# Funciones para la generación de features, y para la intersección de índices entre features y targets.

def make_lag_features(df, lags):
    df_aux = df.copy()
    for lag in lags:
        df_aux[f"lag_{lag}"] = df_aux["value"].shift(lag)
    return df_aux.drop(columns=["value"]).dropna()

def make_rolling_features(df, windows):
    df_aux = df.copy()
    for w in windows:
        df_aux[f"rolling_mean_{w}"] = df_aux["value"].rolling(w).mean()
        df_aux[f"rolling_std_{w}"] = df_aux["value"].rolling(w).std()
        df_aux[f"rolling_min_{w}"] = df_aux["value"].rolling(w).min()
        df_aux[f"rolling_max_{w}"] = df_aux["value"].rolling(w).max()
    return df_aux.drop(columns=["value"]).dropna()

def make_temporal_features(df):
    df_aux = df.copy()
    df_aux["hour"] = df_aux.index.hour
    df_aux["dayofweek"] = df_aux.index.dayofweek
    df_aux["dayofmonth"] = df_aux.index.day
    df_aux["weekofyear"] = df_aux.index.isocalendar().week.astype(int)
    df_aux["month"] = df_aux.index.month
    return df_aux.drop(columns=["value"])

def index_intersection(features, target):
    idx_common = features.index.intersection(target.index)
    return features.loc[idx_common], target.loc[idx_common]

In [ ]:
# Métricas de evaluación para los modelos de forecasting.
def metricas(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "MSE": mean_squared_error(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    }

In [ ]:
lags_base = range(0, 13)  # lags de 0 a 12 horas.
windows_base = [24, 168] # rolling windows de 1 dia y 1 semana.

dta_lagf = make_lag_features(dta, lags_base)
dta_rollf = make_rolling_features(dta, windows_base)
dta_temporalf = make_temporal_features(dta)[["hour", "dayofweek", "dayofmonth"]] #Fijarse que solo se utilizan algunas de las features que proporcionan la funcion.

dta_features = pd.concat([dta_lagf, dta_rollf, dta_temporalf], axis=1).dropna()
dta_features.head()

### RESPONDER

Las celdas anteriores generan las features necesarias para entrenar los modelos de árboles. Discutir:

1. ¿Cómo cambia la cantidad de filas antes (dta) y después de la generación de las features (dta_features)? ¿Por qué?

**Respuestas**



In [ ]:
#TODO: Comparar la cantidad de filas.

In [ ]:
# Se emparejan los features y los targets para que tengan el mismo índice temporal.
# Se divide el conjunto en train y test para ambos problemas de forecasting (1h y 1w).

dta_1h_features, dta_1h_target = index_intersection(dta_features, target_1h)
dta_1w_features, dta_1w_target = index_intersection(dta_features, target_1w)

split_train_end = "2020-05-31"
split_test_start = "2020-06-01"
split_test_end = "2020-06-30"

dta_1h_features_train = dta_1h_features[:split_train_end]
dta_1h_target_train = dta_1h_target[:split_train_end]
dta_1h_features_test = dta_1h_features[split_test_start:split_test_end]
dta_1h_target_test = dta_1h_target[split_test_start:split_test_end]

dta_1w_features_train = dta_1w_features[:split_train_end]
dta_1w_target_train = dta_1w_target[:split_train_end]
dta_1w_features_test = dta_1w_features[split_test_start:split_test_end]
dta_1w_target_test = dta_1w_target[split_test_start:split_test_end]

print(dta_1h_features_train.shape, dta_1h_features_test.shape)
print(dta_1w_features_train.shape, dta_1w_features_test.shape)

## Parte 3 - Forecasting con RandomForest y XGBoost

### Forecasting 1 hora con RandomForest

In [ ]:
# Se instancia un modelo de RF para el problema de forecasting de 1 hora.
rf_1h = RandomForestRegressor(
    n_estimators=50,
    max_leaf_nodes=500,
    random_state=SEED,
    n_jobs=-1
)

# Entrenamiento
t0 = time.perf_counter()
rf_1h.fit(dta_1h_features_train.values, dta_1h_target_train.values.ravel())
rf_1h_train_time = time.perf_counter() - t0

print(f"Tiempo de entrenamiento RF 1h: {rf_1h_train_time:.2f} segundos")

# Predicción
pred_rf_1h = rf_1h.predict(dta_1h_features_test.values)

In [ ]:
metrics_rf_1h = metricas(dta_1h_target_test.values.ravel(), pred_rf_1h.ravel())
metrics_rf_1h

In [ ]:
plt.figure(figsize=(16, 3))
plt.plot(dta_1h_target_test.index[:24 * 10], dta_1h_target_test.values[:24 * 10], label="Test", color="black")
plt.plot(dta_1h_target_test.index[:24 * 10], pred_rf_1h[:24 * 10], label="RandomForest", color="tab:blue")
plt.title("Prediccion 1h - primer tramo de test")
plt.legend(loc="best")
plt.show()

In [ ]:
# Feature importance
fi_rf_1h = pd.Series(rf_1h.feature_importances_, index=dta_1h_features_train.columns).sort_values(ascending=False)
display(fi_rf_1h.head(15).to_frame("importance"))

fi_rf_1h.head(15).sort_values().plot(kind="barh", color="#1f77b4")
plt.title("Top 15 feature importance - RF 1h")
plt.tight_layout()
plt.show()

### Forecasting 1 semana con Random Forest

In [ ]:
# Se instancia un modelo de RF para el problema de forecasting de 1 semana.
rf_1w = RandomForestRegressor(
    n_estimators=50,
    max_leaf_nodes=500,
    random_state=SEED,
    n_jobs=-1
)

# Entrenamiento
t0 = time.perf_counter()
rf_1w.fit(dta_1w_features_train.values, dta_1w_target_train.values)
rf_1w_train_time = time.perf_counter() - t0

print(f"Tiempo de entrenamiento RF 1w: {rf_1w_train_time:.2f} segundos")

# Predicción
pred_rf_1w = rf_1w.predict(dta_1w_features_test.values)

In [ ]:
# Evaluamos solo anclas semanales para comparar ventanas completas
metrics_rf_1w = metricas(dta_1w_target_test.values[::168], pred_rf_1w[::168])
metrics_rf_1w

In [ ]:
# Feature importance
fi_rf_1w = pd.Series(rf_1w.feature_importances_, index=dta_1w_features_train.columns).sort_values(ascending=False)
display(fi_rf_1w.head(15).to_frame("importance"))

fi_rf_1w.head(15).sort_values().plot(kind="barh", color="#1f77b4")
plt.title("Top 15 feature importance - RF 1w")
plt.tight_layout()
plt.show()

In [ ]:
# Visualizacion de la primera ventana semanal pronosticada por RF
inicio = dta_1h_target_test.index[0] + pd.Timedelta(hours=1)
fin = inicio + pd.Timedelta(hours=24 * 7)

y_real = dta_1h_target_test.loc[inicio:fin].values.ravel()
x_real = dta_1h_target_test.loc[inicio:fin].index
y_pred = pred_rf_1w[0].ravel()

n = min(len(x_real), len(y_real), len(y_pred))

plt.figure(figsize=(16, 3))
plt.plot(x_real[:n], y_real[:n], label="Test", color="black")
plt.plot(x_real[:n], y_pred[:n], label="RandomForest 1w", color="tab:blue")
plt.title("Prediccion 1w - primera semana de test")
plt.legend(loc="best")
plt.show()

### DISCUTIR

1. Como cambian los errores al pasar de 1h a 1w?
2. La velocidad de entrenamiento acompaña esa dificultad?

### Forecasting XGBoost

### COMPLETAR Y DISCUTIR:
Repita las celdas anteriores de predicción de la siguiente hora y de la siguiente semana usando XGBoost. Compare los resultados con los obtenidos con RandomForest y discuta:

- Tiempo de entrenamiento.
- Resultados de las métricas de error.

### Forecasting 1 hora con XGBoost

In [ ]:
xgb_1h = xgb.XGBRegressor(
    n_estimators=50,
    max_leaf_nodes=500,
    random_state=SEED,
    objective="reg:squarederror",
    n_jobs=-1
)

#TODO Entrenamiento, y predicción

In [ ]:
#TODO: Feature importance para xgb_1h

In [ ]:
#TODO: Feature importance para xgb_1h

In [ ]:
#TODO: Plot de la primer semana de test pronosticada por XGBoost

### Forecasting 1 semana XGBoost

In [ ]:
xgb_1w = xgb.XGBRegressor(
    n_estimators=50,
    max_leaf_nodes=500,
    random_state=SEED,
    objective="reg:squarederror",
    n_jobs=-1
)

#TODO: Entrenamiento, y predicción

In [ ]:
#TODO: Calcular métricas de desempeño

In [ ]:
#TODO: Feature importance para xgb_1w

In [ ]:
#TODO: Visualizacion de la primera ventana semanal pronosticada por XGBoost

## Parte 4 - EXPERIMENTAR con diferentes features

Utilizando las funciones de generación de features, probar agregar o quitar features que le parezcan relevantes para mejorar el desempeño de los modelos anteriores. Por ejemplo, agregar otras lags o features temporales.



DISCUTIR:
1. Como impactaron en el desempeño del forecasting.
2. Como cambiaron las conclusiones sobre la importancia de las features tanto para el problema de 1h como para el de 1w.
3. Como impactaron en el tiempo de entrenamiento.

In [ ]:
#TODO: completar la generación de features.

dta_lagf_dif = 
dta_rollf_dif = 
dta_temp_dif = 

dta_features_dif = pd.concat([dta_lagf_dif, dta_rollf_dif, dta_temp_dif], axis=1).dropna()
dta_1h_features_dif, dta_1h_target_dif = index_intersection(dta_features_dif, target_1h)
dta_1w_features_dif, dta_1w_target_dif = index_intersection(dta_features_dif, target_1w)

dta_1h_features_dif_train = dta_1h_features_dif[:split_train_end]
dta_1h_target_dif_train = dta_1h_target_dif[:split_train_end]
dta_1h_features_dif_test = dta_1h_features_dif[split_test_start:split_test_end]
dta_1h_target_dif_test = dta_1h_target_dif[split_test_start:split_test_end]

dta_1w_features_dif_train = dta_1w_features_dif[:split_train_end]
dta_1w_target_dif_train = dta_1w_target_dif[:split_train_end]
dta_1w_features_dif_test = dta_1w_features_dif[split_test_start:split_test_end]
dta_1w_target_dif_test = dta_1w_target_dif[split_test_start:split_test_end]

print(dta_1h_features_dif_train.shape, dta_1h_features_dif_test.shape)
print(dta_1w_features_dif_train.shape, dta_1w_features_dif_test.shape)

In [ ]:
# TODO: Entrenar y evaluar modelos de RF y XGBoost con las nuevas features, y comparar resultados con los obtenidos previamente.

## Parte 5 - Deep Learning: RNN vs CNN 1D

Usamos NeuralForecast con horizon 1w (168 pasos) para comparar RNN y CNN 1D (TCN).

In [ ]:
dta_train = dta[:split_train_end].copy()
dta_test = dta[split_test_start:split_test_end].copy()

# Formato largo requerido por NeuralForecast
dta_nf_train = dta_train.reset_index().rename(columns={"timestamp": "ds", "value": "y"})
dta_nf_train["unique_id"] = "MER"

def count_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))

dta_nf_train.head()

In [ ]:
# Definición de los modelos con sus hiperparámetros.

H = 168 # Se define la ventana de predicción como 1 semana (168 horas).
INPUT_SIZE = 2 * 168 # Se define la ventana de entrada como 2 semanas (336 horas).

rnn_model = RNN(
    h=H,
    input_size=INPUT_SIZE,
    inference_input_size=None,
    scaler_type="standard",
    loss=MSE(),
    valid_loss=MSE(),
    encoder_n_layers=2,
    encoder_hidden_size=32,
    max_steps=500
)

tcn_model = TCN(
    h=H,
    input_size=INPUT_SIZE,
    scaler_type="standard",
    loss=MSE(),
    valid_loss=MSE(),
    kernel_size=2,
    dilations=[1, 2, 4],
    encoder_hidden_size=32,
    max_steps=500,
    
)

fcst_rnn = NeuralForecast(models=[rnn_model], freq="h")
fcst_tcn = NeuralForecast(models=[tcn_model], freq="h")

In [ ]:
# Entrenamiento del modelo RNN

t0 = time.perf_counter()
fcst_rnn.fit(df=dta_nf_train, val_size=720 * 3)
rnn_train_time = time.perf_counter() - t0

print(f"Tiempo entrenamiento RNN: {rnn_train_time:.2f}s")

print("Parametros entrenables RNN:", count_parameters(fcst_rnn.models[0]))

In [ ]:
# Entrenamiento del modelo TCN

t0 = time.perf_counter()
fcst_tcn.fit(df=dta_nf_train, val_size=720 * 3)
tcn_train_time = time.perf_counter() - t0

print(f"Tiempo entrenamiento TCN: {tcn_train_time:.2f}s")

print("Parametros entrenables TCN:", count_parameters(fcst_tcn.models[0]))


In [ ]:
# Predicciones de las primeras semanas del conjunto de test.

def rolling_weekly_predict(fcst_model, serie, model_name, start="2020-06-01 00:00:00+00:00", h=168, input_size=2 * 168):
    i = serie.index.get_loc(start)
    forecasts_to_compare = pd.DataFrame()

    for m in range(4):
        df = serie.iloc[max(0, i + (h * m) - input_size): i + (h * m)].copy()
        df_nf = df.reset_index().rename(columns={"timestamp": "ds", "value": "y"})
        df_nf["unique_id"] = "MER"
        forecast = fcst_model.predict(df=df_nf)
        forecasts_to_compare = pd.concat([forecasts_to_compare, forecast], ignore_index=False)

    forecasts_to_compare = forecasts_to_compare.set_index("ds").sort_index()
    return forecasts_to_compare[[model_name]]

pred_rnn_june = rolling_weekly_predict(fcst_rnn, dta, "RNN", h=H, input_size=INPUT_SIZE)
pred_tcn_june = rolling_weekly_predict(fcst_tcn, dta, "TCN", h=H, input_size=INPUT_SIZE)


print(pred_rnn_june.head())
print(pred_tcn_june.head())

In [ ]:
june_test = dta[split_test_start:split_test_end].copy()

idx_rnn = june_test.index.intersection(pred_rnn_june.index)
metrics_rnn_1w = metricas(june_test.loc[idx_rnn, "value"].values, pred_rnn_june.loc[idx_rnn, "RNN"].values)

idx_tcn = june_test.index.intersection(pred_tcn_june.index)
metrics_tcn_1w = metricas(june_test.loc[idx_tcn, "value"].values, pred_tcn_june.loc[idx_tcn, "TCN"].values)

In [ ]:
plt.figure(figsize=(16, 3))
plt.plot(june_test.index[:24 * 14], june_test["value"].values[:24 * 14], label="Test", color="black")
plt.plot(pred_rnn_june.index[:24 * 14], pred_rnn_june["RNN"].values[:24 * 14], label="RNN", alpha=0.9)
plt.plot(pred_tcn_june.index[:24 * 14], pred_tcn_june["TCN"].values[:24 * 14], label="TCN", alpha=0.9)
plt.title("Comparacion RNN vs TCN (inicio de junio)")
plt.legend(loc="best")
plt.show()

In [ ]:
# Comparación global para el problema de forecasting de 1 semana.

pd.DataFrame([
    {"modelo": "RF_1w", "MAE": metrics_rf_1w["MAE"], "MAPE": metrics_rf_1w["MAPE"], "MSE": metrics_rf_1w["MSE"], "train_time": rf_1w_train_time},
    {"modelo": "XGB_1w", "MAE": metrics_xgb_1w["MAE"], "MAPE": metrics_xgb_1w["MAPE"], "MSE": metrics_xgb_1w["MSE"], "train_time": xgb_1w_train_time},
    {"modelo": "RNN_1w", "MAE": metrics_rnn_1w["MAE"], "MAPE": metrics_rnn_1w["MAPE"], "MSE": metrics_rnn_1w["MSE"], "train_time": rnn_train_time},
    {"modelo": "TCN_1w", "MAE": metrics_tcn_1w["MAE"], "MAPE": metrics_tcn_1w["MAPE"], "MSE": metrics_tcn_1w["MSE"], "train_time": tcn_train_time},
])

### DISCUTIR:
1. Que diferencias se observan entre los resultados de RNN y TCN?
2. Discuta los resultados globales de todos los modelos. ¿Cuál es el mejor modelo? ¿Por qué?

### EXPERIMENTAR:
1. Verificar que el largo de la secuencia de entrada no varía la cantidad de parámetros de los modelos de RNN y TCN.
2. Probar diferentes configuraciones de hiperparámetros para RNN y TCN, y discutir como cambian los resultados y el tiempo de entrenamiento.

In [ ]:
#TODO